# MiMo-V2.5-ASR Colab export for Mazda ASR benchmark

Run this notebook in Google Colab with a GPU runtime. It exports rows matching `benchmark/runner.py` for later import into the local reports as service `xiaomi_mimo_2.5`.

Expected inputs:
- A Mazda audio archive, extracted so paths in the manifest are reachable.
- A manifest CSV with columns: `dataset`, `sample_id`, `locale`, `reference`, `audio_path`, and optionally `duration_s`.


In [ ]:
!nvidia-smi
!python --version

In [ ]:
!git clone https://github.com/XiaomiMiMo/MiMo-V2.5-ASR.git
%cd MiMo-V2.5-ASR
!pip install -r requirements.txt
!pip install huggingface-hub jiwer soundfile
# Colab flash-attn builds can be slow; install the repo-recommended version if your runtime supports it.
!pip install flash-attn==2.7.4.post1 --no-build-isolation

In [ ]:
!hf download XiaomiMiMo/MiMo-Audio-Tokenizer --local-dir ./models/MiMo-Audio-Tokenizer
!hf download XiaomiMiMo/MiMo-V2.5-ASR --local-dir ./models/MiMo-V2.5-ASR

## Upload the Mazda package

First generate the package locally with `scripts/export_mimo_colab_package.py`, then upload `mimo_colab_package.zip` here. The notebook extracts it and uses its `manifest.csv`.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
package_dir = '/content/mimo_colab_package'
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(package_dir)
manifest_path = f'{package_dir}/manifest.csv'
manifest_path

In [ ]:
import csv
import re
import string
import time
from pathlib import Path

import soundfile as sf
from jiwer import wer as jiwer_wer
from src.mimo_audio.mimo_audio import MimoAudio

SERVICE = 'xiaomi_mimo_2.5'
CSV_FIELDS = [
    'dataset', 'sample_id', 'service', 'locale', 'duration_s',
    'reference', 'hypothesis',
    'wer', 'sentence_error',
    'ins', 'del', 'sub', 'ref_len',
    'first_latency_ms', 'lbl_ms',
    'upl_ms', 'upl_self_ms', 'upl_anchor',
    'speech_start_s', 'speech_end_s', 'boundary_fix_action',
    'first_word_start_s', 'last_word_end_s',
    'vad_truncated_s',
    'error',
]

def normalize_text(text):
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def edit_counts(ref, hyp):
    ref_words = normalize_text(ref).split()
    hyp_words = normalize_text(hyp).split()
    n, m = len(ref_words), len(hyp_words)
    dp = [[(0, 0, 0, 0) for _ in range(m + 1)] for _ in range(n + 1)]
    for i in range(1, n + 1):
        c, ins, dele, sub = dp[i - 1][0]
        dp[i][0] = (c + 1, ins, dele + 1, sub)
    for j in range(1, m + 1):
        c, ins, dele, sub = dp[0][j - 1]
        dp[0][j] = (c + 1, ins + 1, dele, sub)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            choices = []
            if ref_words[i - 1] == hyp_words[j - 1]:
                choices.append(dp[i - 1][j - 1])
            else:
                c, ins, dele, sub = dp[i - 1][j - 1]
                choices.append((c + 1, ins, dele, sub + 1))
            c, ins, dele, sub = dp[i][j - 1]
            choices.append((c + 1, ins + 1, dele, sub))
            c, ins, dele, sub = dp[i - 1][j]
            choices.append((c + 1, ins, dele + 1, sub))
            dp[i][j] = min(choices, key=lambda x: x[0])
    _, ins, dele, sub = dp[n][m]
    return ins, dele, sub, n

model = MimoAudio(
    model_path='./models/MiMo-V2.5-ASR',
    tokenizer_path='./models/MiMo-Audio-Tokenizer',
)

In [ ]:
rows = []
with open(manifest_path, encoding='utf-8-sig', newline='') as f:
    manifest_rows = list(csv.DictReader(f))

base_dir = Path(manifest_path).parent
for item in manifest_rows:
    audio_path = Path(item['audio_path'])
    if not audio_path.is_absolute():
        audio_path = base_dir / audio_path
    row = {field: '' for field in CSV_FIELDS}
    row.update({
        'dataset': item['dataset'],
        'sample_id': item['sample_id'],
        'service': SERVICE,
        'locale': item['locale'],
        'reference': item['reference'],
    })
    try:
        if item.get('duration_s'):
            row['duration_s'] = item['duration_s']
        else:
            info = sf.info(str(audio_path))
            row['duration_s'] = round(info.frames / info.samplerate, 3)
        t0 = time.perf_counter()
        row['hypothesis'] = model.asr_sft(str(audio_path), audio_tag='<english>')
        row['lbl_ms'] = round((time.perf_counter() - t0) * 1000, 1)
        ref_norm = normalize_text(row['reference'])
        hyp_norm = normalize_text(row['hypothesis'])
        row['wer'] = round(jiwer_wer(ref_norm, hyp_norm), 4)
        row['sentence_error'] = int(ref_norm != hyp_norm)
        ins, dele, sub, ref_len = edit_counts(row['reference'], row['hypothesis'])
        row['ins'] = ins
        row['del'] = dele
        row['sub'] = sub
        row['ref_len'] = ref_len
    except Exception as exc:
        row['error'] = repr(exc)
    rows.append(row)

out_path = 'mimo_2_5_export.csv'
with open(out_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    writer.writerows(rows)

print(f'wrote {len(rows)} rows to {out_path}')
files.download(out_path)